In [0]:
"""
05_fact_production.py

Manufacturing Production Fact Table

Business Grain:
    One manufacturing execution.

Sources:
    execution_events
    work_order_events
    product_dimension

Target:
    fact_production

Author:
Sumanth Vempalle

Version:
2.3.0
"""

import dlt

from pyspark.sql.functions import col


# ============================================================
# Production Fact Table
# ============================================================

@dlt.table(
    name="fact_production",
    comment="Manufacturing Production Fact Table.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def fact_production():

    executions = (

        dlt.read(
            "execution_events"
        )

        # One execution per business key
        .dropDuplicates(
            ["execution_id"]
        )

    )

    work_orders = (

        dlt.read(
            "work_order_events"
        )

        # One work order per business key
        .dropDuplicates(
            ["work_order_id"]
        )

    )

    products = dlt.read(
        "product_dimension"
    )

    return (

        executions.alias("ex")

        .join(

            work_orders.alias("wo"),

            on="work_order_id",

            how="left",

        )

        .join(

            products.alias("pd"),

            on="product_code",

            how="left",

        )

        .select(

            # ====================================================
            # Event
            # ====================================================

            col("ex.event_id"),

            col("ex.event_timestamp"),

            col("ex.event_version"),

            # ====================================================
            # Manufacturing Keys
            # ====================================================

            col("ex.plant_code"),

            col("ex.execution_id"),

            col("ex.work_order_id"),

            col("ex.product_code"),

            # ====================================================
            # Product Dimension
            # ====================================================

            col("pd.product_name"),

            col("pd.family"),

            col("pd.rated_voltage_kv"),

            col("pd.routing_version"),

            # ====================================================
            # Work Order
            # ====================================================

            col("wo.sap_order_number"),

            col("wo.quantity"),

            col("wo.priority"),

            col("wo.planned_shift"),

            col("wo.status"),

            # ====================================================
            # Execution
            # ====================================================

            col("ex.production_line"),

            # ====================================================
            # Audit
            # ====================================================

            col("ex.silver_processing_timestamp"),

        )

    )